# Session 05 — Object Detection and Shape Analysis

**Goal:** use contours and simple geometry to count, classify, and measure simple shapes. This is a rule-based method: it works when the objects are clear and separate.


## 1. Get the drawing tools

OpenCV finds boundaries. NumPy stores the image as numbers. Matplotlib shows the result.


In [ ]:
# Colab already includes OpenCV, NumPy, and Matplotlib.
# Import the image tools we will use.
import cv2
import numpy as np
import matplotlib.pyplot as plt


## 2. Make a controlled test image

We start with a clear circle, rectangle, and triangle. This lets us see exactly what a contour rule can do.


In [ ]:
# This lesson begins with a controlled picture.
# Controlled means we know what should be found before the code runs.

# Make a blank white image.
# 420 is the height, 640 is the width, and 3 stores blue, green, and red colour values.
image = np.full((420, 640, 3), 255, dtype=np.uint8)

# Draw a filled black circle.
# The final -1 tells OpenCV to fill the circle instead of drawing only its edge.
cv2.circle(image, (120, 210), 65, (0, 0, 0), -1)

# Draw a filled black rectangle using its top-left and bottom-right corners.
cv2.rectangle(image, (260, 145), (400, 275), (0, 0, 0), -1)

# Store the triangle's three corner positions as a small list of points.
triangle = np.array([[500, 285], [565, 145], [620, 285]])

# Draw and fill the triangle from those three points.
cv2.fillPoly(image, [triangle], (0, 0, 0))

# Convert the colour image into grayscale.
# A grayscale pixel has one brightness value, which makes thresholding easier.
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Set the cut-off brightness.
# Pixels darker than this become white in our mask because we use BINARY_INV.
# Try changing only this value later to test how a contour rule can fail.
THRESHOLD = 127

# Make a binary mask: every pixel becomes either black (background) or white (object).
# The underscore stores a value we do not need; binary is the useful black-and-white result.
_, binary = cv2.threshold(gray, THRESHOLD, 255, cv2.THRESH_BINARY_INV)

# Create a figure with two display spaces side by side.
plt.figure(figsize=(12, 4))

# Display space 1: show the original drawing.
plt.subplot(1, 2, 1)
# OpenCV stores colours as BGR, while Matplotlib expects RGB, so swap the order for display.
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Original drawing")
plt.axis("off")

# Display space 2: show the binary mask in black and white.
plt.subplot(1, 2, 2)
plt.imshow(binary, cmap="gray")
plt.title("Binary mask")
plt.axis("off")

# Put both pictures on the screen.
plt.show()


## 3. Find, label, and measure each contour

A contour is a list of points around one connected boundary. We use the number of simplified corners as a small rule for the shape name.


In [ ]:
# Ask OpenCV to find the outside boundary of every connected white region.
# RETR_EXTERNAL ignores holes inside an object because we only want each object's outer edge.
# CHAIN_APPROX_SIMPLE avoids saving unnecessary repeated edge points.
contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Copy the original drawing.
# We will draw evidence on this copy, leaving the original image unchanged.
result = image.copy()

# Work through one detected contour at a time.
for contour in contours:
    # Measure the number of pixels inside this boundary.
    area = cv2.contourArea(contour)

    # Measure the distance around this boundary.
    # True means the final boundary point connects back to the first point.
    perimeter = cv2.arcLength(contour, True)

    # Simplify the detailed curve into major turns or corners.
    # 0.04 means we allow a small amount of detail to be removed.
    simple_shape = cv2.approxPolyDP(contour, 0.04 * perimeter, True)

    # Count the remaining corner points.
    corners = len(simple_shape)

    # Apply a deliberately simple rule to the corner count.
    # Three corners make a triangle.
    if corners == 3:
        name = "triangle"
    # Four corners make a rectangle in this controlled picture.
    elif corners == 4:
        name = "rectangle"
    # Any other count is called a circle here.
    # This works for this picture, but it would be too weak for many real objects.
    else:
        name = "circle"

    # Find the smallest upright rectangle that contains this contour.
    x, y, width, height = cv2.boundingRect(contour)

    # Draw that detection box in blue on the result image.
    cv2.rectangle(result, (x, y), (x + width, y + height), (21, 138, 194), 2)

    # Make a text label that shows both the shape decision and its measured area.
    label = f"{name}: {area:.0f} px"

    # Draw the label just above the detection box in orange.
    cv2.putText(result, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (246, 138, 69), 2)

# Count how many connected objects OpenCV found.
print("Objects counted:", len(contours))

# Create a larger display space for the inspected result.
plt.figure(figsize=(12, 6))
# Convert from OpenCV's BGR order to the RGB order Matplotlib displays correctly.
plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.axis("off")
# Show the contour evidence, labels, and boxes.
plt.show()


## 4. Stress-test the rule

Change only `THRESHOLD`, rerun the notebook, and write down what changes. Then answer: would this rule still work if objects overlapped, were rotated, or had a busy background?

**Evidence record:**
- Shape rule used: ______
- Objects counted: ______
- First condition that breaks it: ______
- When a deep detector would be more useful: ______
